# Experiment: Challenge A regression dataset

Objective:
- Build leakage-safe modeling tables from the PDE CSV.
- Predict `target_refill_gap_days` for regression rows with an observed next fill.
- Build a censoring-aware `target_is_late_refill_7d` label for classification.


In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

GRACE_DAYS = 7
ROOT = Path.cwd()
RAW_CSV = ROOT / "Dataset" / "DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1" / "DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv"
OUTPUT_DIR = ROOT / "DATA-AI-Hackathon-Track-1" / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV, OUTPUT_DIR


(WindowsPath('g:/My Drive/Personal Drive 2026/hackaton lida 2026/Dataset/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv'),
 WindowsPath('g:/My Drive/Personal Drive 2026/hackaton lida 2026/DATA-AI-Hackathon-Track-1/data/processed'))

## 1. Load the raw PDE data

We keep identifiers as text so leading zeros are preserved, especially for `PROD_SRVC_ID`.


In [2]:
dtype_map = {
    "DESYNPUF_ID": "string",
    "PDE_ID": "string",
    "SRVC_DT": "Int32", # convert to date
    "PROD_SRVC_ID": "string", 
    "QTY_DSPNSD_NUM": "float32",
    "DAYS_SUPLY_NUM": "Int16",
    "PTNT_PAY_AMT": "float32",
    "TOT_RX_CST_AMT": "float32",
}

pde = pd.read_csv(RAW_CSV, dtype=dtype_map)
pde["service_date"] = pd.to_datetime(
    pde["SRVC_DT"].astype("string"),
    format="%Y%m%d",
    errors="coerce",
)

pde = pde.sort_values(
    ["DESYNPUF_ID", "PROD_SRVC_ID", "service_date", "PDE_ID"],
    kind="mergesort",
).reset_index(drop=True)

print(f"Rows: {len(pde):,}")
print(f"Patients: {pde['DESYNPUF_ID'].nunique():,}")
print(f"Products: {pde['PROD_SRVC_ID'].nunique():,}")
pde.head()


Rows: 5,552,421
Patients: 99,538
Products: 268,563


,DESYNPUF_ID,PDE_ID,SRVC_DT,PROD_SRVC_ID,QTY_DSPNSD_NUM,DAYS_SUPLY_NUM,PTNT_PAY_AMT,TOT_RX_CST_AMT,service_date
0,00013D2EFD8E45D1,233854493690818,20100901,00006004582,30.0,30,50.0,150.0,2010-09-01
1,00013D2EFD8E45D1,233754493779271,20081221,00006071282,30.0,30,0.0,10.0,2008-12-21
2,00013D2EFD8E45D1,233514494068331,20080410,00006073087,30.0,90,10.0,100.0,2008-04-10
3,00013D2EFD8E45D1,233944491253625,20090201,00006074928,30.0,90,30.0,60.0,2009-02-01
4,00013D2EFD8E45D1,233594491623171,20080125,00008032506,180.0,30,0.0,10.0,2008-01-25


## 2. Normalize NDC and choose the drug grouping level

`PROD_SRVC_ID` is already an NDC in CMS 11-digit format. We derive:
- `ndc11`: exact dispensed package-level code
- `ndc9`: product-level code without the package segment

For refill modeling, `ndc11` is often too granular because package changes split the same therapy into separate histories. We therefore compare both and default to `ndc9` as the grouping key.


In [ ]:
pde["ndc11"] = pde["PROD_SRVC_ID"].astype("string").str.strip().str.zfill(11)
pde["labeler_code"] = pde["ndc11"].str[:5]
pde["product_code"] = pde["ndc11"].str[5:9]
pde["package_code"] = pde["ndc11"].str[9:11]

pde["ndc11_hyphen"] = (
    pde["labeler_code"] + "-" + pde["product_code"] + "-" + pde["package_code"]
)
pde["ndc9"] = pde["labeler_code"] + pde["product_code"]
pde["ndc9_hyphen"] = pde["labeler_code"] + "-" + pde["product_code"]


def summarize_grouping(code_col: str) -> dict[str, float]:
    pair_sizes = pde.groupby(["DESYNPUF_ID", code_col], sort=False)["PDE_ID"].size()
    return {
        "patient_drug_pairs": int(len(pair_sizes)),
        "share_with_1_fill": float((pair_sizes == 1).mean()),
        "share_with_gt1_fill": float((pair_sizes > 1).mean()),
        "max_fills_in_pair": int(pair_sizes.max()),
    }


grouping_comparison = pd.DataFrame(
    {
        "ndc11_exact_package": summarize_grouping("ndc11"),
        "ndc9_product": summarize_grouping("ndc9"),
    }
).T

DRUG_GROUP_COL = "ndc9"

print(f"Selected grouping key: {DRUG_GROUP_COL}")
grouping_comparison


## 3. Build targets and chronology

We build refill histories at the chosen drug grouping level stored in `drug_group_key`. By default this is `ndc9`, which is less sparse than the exact package-level `ndc11`.


In [ ]:
pde["drug_group_key"] = pde[DRUG_GROUP_COL]

group_cols = ["DESYNPUF_ID", "drug_group_key"]
group_sizes = pde.groupby(group_cols, sort=False)["PDE_ID"].transform("size")
grouped = pde.groupby(group_cols, sort=False)

pde["fill_sequence_number"] = grouped.cumcount().add(1).astype("int32")
pde["fills_in_patient_drug_history"] = group_sizes.astype("int32")
pde["prior_fill_count"] = (pde["fill_sequence_number"] - 1).astype("int32")

# main column
pde["run_out_date"] = pde["service_date"] + pd.to_timedelta(
    pde["DAYS_SUPLY_NUM"].fillna(0).astype("int32"), unit="D"
)
pde.loc[
    pde["service_date"].isna() | pde["DAYS_SUPLY_NUM"].isna(),
    "run_out_date",
] = pd.NaT

pde["previous_service_date"] = grouped["service_date"].shift(1)
pde["previous_run_out_date"] = grouped["run_out_date"].shift(1)
pde["previous_days_supply"] = grouped["DAYS_SUPLY_NUM"].shift(1)
pde["previous_quantity"] = grouped["QTY_DSPNSD_NUM"].shift(1)
pde["previous_patient_pay"] = grouped["PTNT_PAY_AMT"].shift(1)
pde["previous_total_cost"] = grouped["TOT_RX_CST_AMT"].shift(1)

pde["next_fill_date"] = grouped["service_date"].shift(-1)
pde["target_days_until_next_fill"] = (pde["next_fill_date"] - pde["service_date"]).dt.days
pde["target_refill_gap_days"] = (pde["next_fill_date"] - pde["run_out_date"]).dt.days
pde["gap_from_previous_fill_days"] = (pde["service_date"] - pde["previous_run_out_date"]).dt.days

data_end = pde["service_date"].max()
pde["late_refill_deadline"] = pde["run_out_date"] + pd.to_timedelta(GRACE_DAYS, unit="D")
pde["late_refill_label_observable"] = (
    pde["run_out_date"].notna() & (pde["late_refill_deadline"] <= data_end)
)

pde["has_next_fill"] = pde["next_fill_date"].notna()
pde["has_previous_fill"] = pde["previous_service_date"].notna()
pde["target_is_late_refill_7d"] = (
    pde["has_next_fill"] & (pde["target_refill_gap_days"] > GRACE_DAYS)
).astype("int8")

pde[[
    "DESYNPUF_ID",
    "drug_group_key",
    "PROD_SRVC_ID",
    "ndc11",
    "ndc9",
    "service_date",
    "run_out_date",
    "next_fill_date",
    "late_refill_label_observable",
    "gap_from_previous_fill_days",
    "target_refill_gap_days",
    "target_is_late_refill_7d",
]].head(50)


In [4]:
# Diagnose why prior_fill_count is often 0
pair_history_counts = (
    pde.groupby(["DESYNPUF_ID", "drug_group_key"], sort=False)["PDE_ID"]
    .size()
    .rename("fills_per_patient_drug")
)

history_diagnostic = (
    pair_history_counts.value_counts()
    .sort_index()
    .rename_axis("fills_per_patient_drug")
    .reset_index(name="pair_count")
)
history_diagnostic["pair_share"] = (
    history_diagnostic["pair_count"] / history_diagnostic["pair_count"].sum()
)

print(f"Grouping column used for chronology: {DRUG_GROUP_COL}")
print(f"Patient-drug pairs: {len(pair_history_counts):,}")
print(f"Pairs with only 1 fill: {(pair_history_counts == 1).mean():.4%}")
print(f"Pairs with more than 1 fill: {(pair_history_counts > 1).mean():.4%}")

repeated_pair_preview = pde.loc[
    pde["fills_in_patient_drug_history"] > 1,
    [
        "DESYNPUF_ID",
        "drug_group_key",
        "PROD_SRVC_ID",
        "ndc11",
        "ndc9",
        "service_date",
        "fill_sequence_number",
        "fills_in_patient_drug_history",
        "prior_fill_count",
        "target_refill_gap_days",
    ],
].head(20)

history_diagnostic.head(10), repeated_pair_preview


Patient-drug pairs: 5,549,151
Pairs with only 1 fill: 99.9412%
Pairs with more than 1 fill: 0.0588%


(   fills_per_patient_drug  pair_count  pair_share
 0                       1     5545888    0.999412
 1                       2        3256    0.000587
 2                       3           7    0.000001,
            DESYNPUF_ID PROD_SRVC_ID service_date  fill_sequence_number  \
 211   0001FDD721E223DC  00536358006   2008-02-17                     1   
 212   0001FDD721E223DC  00536358006   2009-11-14                     2   
 348   00036A21B65B0206  49884065501   2008-03-21                     1   
 349   00036A21B65B0206  49884065501   2010-03-06                     2   
 4399  00271F7DF9C2B88A  00006073178   2008-01-29                     1   
 4400  00271F7DF9C2B88A  00006073178   2010-03-28                     2   
 4520  00271F7DF9C2B88A  59762432006   2008-04-24                     1   
 4521  00271F7DF9C2B88A  59762432006   2008-08-14                     2   
 6217  00354528E7D3A756  65084025214   2008-06-14                     1   
 6218  00354528E7D3A756  65084025214   2008-0

## 3b. Analyze `PROD_SRVC_ID` after stripping leading zeros

Before engineering features, we test a simpler hypothesis: maybe the product identifier looks sparse only because of left-zero padding. We compare the raw `PROD_SRVC_ID` against a version with leading zeros removed.

This is only a formatting diagnostic. If the cardinality and refill-history density do not change, then the sparsity problem is not caused by left-zero padding.


In [ ]:
pde["prod_id_raw"] = pde["PROD_SRVC_ID"].astype("string").str.strip()
pde["prod_id_no_left_zeros"] = pde["prod_id_raw"].str.lstrip("0")
pde.loc[pde["prod_id_no_left_zeros"] == "", "prod_id_no_left_zeros"] = "0"

product_id_summary = pd.DataFrame(
    {
        "metric": [
            "unique_ids",
            "patient_product_pairs",
            "share_pairs_with_gt1_fill",
            "share_pairs_with_1_fill",
        ],
        "raw_prod_id": [
            pde["prod_id_raw"].nunique(),
            pde.groupby(["DESYNPUF_ID", "prod_id_raw"], sort=False)["PDE_ID"].ngroups,
            (pde.groupby(["DESYNPUF_ID", "prod_id_raw"], sort=False)["PDE_ID"].size() > 1).mean(),
            (pde.groupby(["DESYNPUF_ID", "prod_id_raw"], sort=False)["PDE_ID"].size() == 1).mean(),
        ],
        "prod_id_no_left_zeros": [
            pde["prod_id_no_left_zeros"].nunique(),
            pde.groupby(["DESYNPUF_ID", "prod_id_no_left_zeros"], sort=False)["PDE_ID"].ngroups,
            (pde.groupby(["DESYNPUF_ID", "prod_id_no_left_zeros"], sort=False)["PDE_ID"].size() > 1).mean(),
            (pde.groupby(["DESYNPUF_ID", "prod_id_no_left_zeros"], sort=False)["PDE_ID"].size() == 1).mean(),
        ],
    }
)

prod_id_mapping = pde[["prod_id_raw", "prod_id_no_left_zeros"]].drop_duplicates()
stripped_id_collisions = (
    prod_id_mapping["prod_id_no_left_zeros"].value_counts().gt(1).sum()
)

prod_id_examples = prod_id_mapping.loc[
    prod_id_mapping["prod_id_raw"] != prod_id_mapping["prod_id_no_left_zeros"]
].head(15)

print(f"IDs collapsed after stripping zeros: {stripped_id_collisions}")
product_id_summary, prod_id_examples


## 4. Create leakage-safe features

These features use only the current fill and earlier history. They do not use `next_fill_date` or any future information.


In [5]:
pde["quantity_per_day"] = pde["QTY_DSPNSD_NUM"] / pde["DAYS_SUPLY_NUM"].replace(0, pd.NA)
pde["patient_cost_share"] = np.where(
    pde["TOT_RX_CST_AMT"] > 0,
    pde["PTNT_PAY_AMT"] / pde["TOT_RX_CST_AMT"],
    np.nan,
)

pde["calendar_year"] = pde["service_date"].dt.year.astype("Int16")
pde["calendar_month"] = pde["service_date"].dt.month.astype("Int8")
pde["calendar_quarter"] = pde["service_date"].dt.quarter.astype("Int8")
pde["service_day_of_week"] = pde["service_date"].dt.dayofweek.astype("Int8")
pde["service_day_of_year"] = pde["service_date"].dt.dayofyear.astype("Int16")

pde["is_30_day_supply"] = (pde["DAYS_SUPLY_NUM"] == 30).astype("int8")
pde["is_90_day_supply"] = (pde["DAYS_SUPLY_NUM"] == 90).astype("int8")

pde["days_since_previous_fill"] = (pde["service_date"] - pde["previous_service_date"]).dt.days
pde["days_supply_change_vs_previous"] = pde["DAYS_SUPLY_NUM"] - pde["previous_days_supply"]
pde["quantity_change_vs_previous"] = pde["QTY_DSPNSD_NUM"] - pde["previous_quantity"]
pde["patient_pay_change_vs_previous"] = pde["PTNT_PAY_AMT"] - pde["previous_patient_pay"]
pde["total_cost_change_vs_previous"] = pde["TOT_RX_CST_AMT"] - pde["previous_total_cost"]

safe_sources = {
    "DAYS_SUPLY_NUM": "prior_mean_days_supply",
    "QTY_DSPNSD_NUM": "prior_mean_quantity",
    "PTNT_PAY_AMT": "prior_mean_patient_pay",
    "TOT_RX_CST_AMT": "prior_mean_total_cost",
}

prior_count_nonzero = pde["prior_fill_count"].replace(0, np.nan)
for source_col, feature_col in safe_sources.items():
    cumulative_sum = grouped[source_col].cumsum()
    prior_sum = cumulative_sum - pde[source_col]
    pde[feature_col] = prior_sum / prior_count_nonzero

feature_preview = [
    "fill_sequence_number",
    "prior_fill_count",
    "quantity_per_day",
    "patient_cost_share",
    "days_since_previous_fill",
    "gap_from_previous_fill_days",
    "prior_mean_days_supply",
    "prior_mean_quantity",
]
pde[feature_preview].head(10)


,fill_sequence_number,prior_fill_count,quantity_per_day,patient_cost_share,days_since_previous_fill,gap_from_previous_fill_days,prior_mean_days_supply,prior_mean_quantity
0,1,0,1.0,0.333333,NaN,NaN,<NA>,NaN
1,1,0,1.0,0.000000,NaN,NaN,<NA>,NaN
2,1,0,0.333333,0.100000,NaN,NaN,<NA>,NaN
3,1,0,0.333333,0.500000,NaN,NaN,<NA>,NaN
4,1,0,6.0,0.000000,NaN,NaN,<NA>,NaN
5,1,0,2.0,NaN,NaN,NaN,<NA>,NaN
6,1,0,1.0,1.000000,NaN,NaN,<NA>,NaN
7,1,0,12.0,0.250000,NaN,NaN,<NA>,NaN
8,1,0,1.0,0.000000,NaN,NaN,<NA>,NaN
9,1,0,0.333333,0.000000,NaN,NaN,<NA>,NaN


## 5. Assemble the regression dataset

`regression_df` keeps identifiers and dates for analysis.
`regression_numeric_df` keeps only numeric features and simple missing-value handling so it can go straight into baseline regressors.


In [ ]:
key_columns = list(dict.fromkeys([
    "DESYNPUF_ID",
    "PDE_ID",
    "drug_group_key",
    DRUG_GROUP_COL,
    "PROD_SRVC_ID",
    "ndc11",
    "ndc9",
    "service_date",
    "run_out_date",
]))

feature_columns = [
    "QTY_DSPNSD_NUM",
    "DAYS_SUPLY_NUM",
    "PTNT_PAY_AMT",
    "TOT_RX_CST_AMT",
    "fill_sequence_number",
    "prior_fill_count",
    "has_previous_fill",
    "quantity_per_day",
    "patient_cost_share",
    "calendar_year",
    "calendar_month",
    "calendar_quarter",
    "service_day_of_week",
    "service_day_of_year",
    "is_30_day_supply",
    "is_90_day_supply",
    "days_since_previous_fill",
    "gap_from_previous_fill_days",
    "previous_days_supply",
    "previous_quantity",
    "previous_patient_pay",
    "previous_total_cost",
    "days_supply_change_vs_previous",
    "quantity_change_vs_previous",
    "patient_pay_change_vs_previous",
    "total_cost_change_vs_previous",
    "prior_mean_days_supply",
    "prior_mean_quantity",
    "prior_mean_patient_pay",
    "prior_mean_total_cost",
]

regression_target_columns = [
    "target_days_until_next_fill",
    "target_refill_gap_days",
    "target_is_late_refill_7d",
]
classification_target_columns = ["target_is_late_refill_7d"]

regression_df = pde.loc[
    pde["has_next_fill"],
    key_columns + feature_columns + regression_target_columns,
].copy()
regression_df["has_previous_fill"] = regression_df["has_previous_fill"].astype("int8")

classification_df = pde.loc[
    pde["has_next_fill"] & pde["late_refill_label_observable"],
    key_columns + feature_columns + regression_target_columns,
].copy()
classification_df["has_previous_fill"] = classification_df["has_previous_fill"].astype("int8")
classification_df["target_is_late_refill_7d"] = classification_df["target_is_late_refill_7d"].astype("int8")

numeric_feature_columns = feature_columns.copy()
regression_numeric_df = regression_df[numeric_feature_columns + regression_target_columns].copy()
regression_numeric_df = regression_numeric_df.fillna(0)

classification_numeric_df = classification_df[
    numeric_feature_columns + classification_target_columns
].copy()
classification_numeric_df = classification_numeric_df.fillna(0)

print(f"Regression rows: {len(regression_df):,}")
print(f"Classification rows: {len(classification_df):,}")
print(
    "Rows excluded from classification for end-of-window censoring: "
    f"{(pde['has_next_fill'] & ~pde['late_refill_label_observable']).sum():,}"
)
classification_df.head()


In [ ]:
dataset_summary = {
    "data_end": str(data_end.date()),
    "regression_rows": int(len(regression_df)),
    "regression_patients": int(regression_df["DESYNPUF_ID"].nunique()),
    "regression_target_refill_gap_days_mean": float(regression_df["target_refill_gap_days"].mean()),
    "regression_target_refill_gap_days_median": float(regression_df["target_refill_gap_days"].median()),
    "classification_rows": int(len(classification_df)),
    "classification_patients": int(classification_df["DESYNPUF_ID"].nunique()),
    "classification_late_refill_rate_7d": float(classification_df["target_is_late_refill_7d"].mean()),
    "classification_rows_excluded_for_censoring": int(
        (pde["has_next_fill"] & ~pde["late_refill_label_observable"]).sum()
    ),
    "classification_max_service_date": str(classification_df["service_date"].max().date()),
}
dataset_summary


## 6. Save the outputs

The first two files are regression-focused. The last two are classification-focused and exclude rows whose late-refill outcome is censored beyond the end of 2010.


In [ ]:
regression_path = OUTPUT_DIR / "challenge_a_regression_dataset.csv"
regression_numeric_path = OUTPUT_DIR / "challenge_a_regression_numeric.csv"
classification_path = OUTPUT_DIR / "challenge_a_classification_dataset.csv"
classification_numeric_path = OUTPUT_DIR / "challenge_a_classification_numeric.csv"

regression_df.to_csv(regression_path, index=False)
regression_numeric_df.to_csv(regression_numeric_path, index=False)
classification_df.to_csv(classification_path, index=False)
classification_numeric_df.to_csv(classification_numeric_path, index=False)

(
    regression_path,
    regression_numeric_path,
    classification_path,
    classification_numeric_path,
)
